In [1]:
import torch
import numpy as np
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import os

In [ ]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="sentence_transformers_bi_encoder_cosine_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [2]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name, device=device)
print("model:", model_name)


device: mps


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model: sentence-transformers/all-MiniLM-L6-v2


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [4]:
batch_size = 128
vault.create_embedding_list("sentence-transformers-sentence-1", ndim=384)
vault.create_embedding_list("sentence-transformers-sentence-2", ndim=384)
emb1_batches = []
emb2_batches = []

for i in tqdm(range(0, len(ds), batch_size), desc="Encoding"):
    batch_s1 = sent1[i:i + batch_size]
    batch_s2 = sent2[i:i + batch_size]

    e1 = model.encode(
        batch_s1,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    e2 = model.encode(
        batch_s2,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    e1_list = e1.cpu().tolist()
    e2_list = e2.cpu().tolist()
    for j in range(len(e1_list)):
        vault.append_embedding("sentence-transformers-sentence-1", e1_list[j], 
                           input_items = {"glue_mrpc_validation": [i*batch_size + j, i*batch_size + j + 1]}
                           )
        vault.append_embedding("sentence-transformers-sentence-2", e2_list[j], 
                           input_items = {"glue_mrpc_validation": [i*batch_size + j, i*batch_size + j + 1]}
                           )
        
    emb1_batches.append(e1)
    emb2_batches.append(e2)

emb1 = torch.cat(emb1_batches, dim=0)
emb2 = torch.cat(emb2_batches, dim=0)

print("emb1 shape:", tuple(emb1.shape))
print("emb2 shape:", tuple(emb2.shape))


Encoding:   0%|          | 0/4 [00:00<?, ?it/s]

emb1 shape: (408, 384)
emb2 shape: (408, 384)


In [ ]:
description = "sentence-transformers-sentence-1 is an embedding dataset containing the encoded representations of the sentence1 text from each example in the GLUE MRPC validation split. It is stored as an embedding list where each entry is a 384-dimensional float vector produced by the sentence-transformers/all-MiniLM-L6-v2 model with normalized embeddings enabled, and each embedding is linked back to the corresponding source item in glue_mrpc_validation. The dataset has one embedding entry per validation example and does not store the raw sentence text itself; its primary field is the embedding vector. In this workflow, it provides the sentence1 side of the bi-encoder representation, which is paired with sentence-transformers-sentence-2 to compute cosine similarity scores and generate paraphrase predictions."
embedding = get_embeddings(description)
vault.create_description("sentence-transformers-sentence-1", description, embedding)

properties = {"type": "text embedding", "task": "paraphrase detection", "role": "sentence1", "source": "glue/mrpc", "split": "validation", "size": "408", "language": "en", "domain": "news", "model": "sentence-transformers/all-MiniLM-L6-v2", "embedding_dim": "384", "normalization": "l2-normalized", "similarity": "cosine"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence-transformers-sentence-1", cat, embedding, prop)

description = "sentence-transformers-sentence-2 is an embedding dataset containing one 384-dimensional normalized sentence embedding for each sentence2 entry in the glue_mrpc_validation dataset. Each item stores the vector produced by the sentence-transformers/all-MiniLM-L6-v2 model, with rows aligned index-by-index to the original validation examples and linked back to the corresponding source record in glue_mrpc_validation. This dataset has a single embedding field per item (384 float values) plus TableVault lineage metadata from the input_items mapping. Its role in the workflow is to represent the second sentence of each MRPC pair in vector form so it can be compared with the matching sentence-transformers-sentence-1 embedding using cosine similarity to generate paraphrase predictions."
embedding = get_embeddings(description)
vault.create_description("sentence-transformers-sentence-2", description, embedding)

properties = {"task": "paraphrase detection", "representation": "sentence embedding", "source": "glue/mrpc", "split": "validation", "source_column": "sentence2", "model": "sentence-transformers/all-MiniLM-L6-v2", "embedding_dim": "384", "similarity_metric": "cosine", "normalization": "L2-normalized", "size": "408", "text_type": "sentence pair second sentence", "framework": "sentence-transformers"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence-transformers-sentence-2", cat, embedding, prop)

In [5]:
threshold = 0.80

similarities = (emb1 * emb2).sum(dim=1)
scores = similarities.detach().cpu().numpy()
y_pred = (scores >= threshold).astype(np.int64)

vault.create_record_list("sentence_transformers_mrpc_prediction", column_names=["prediction"])

for i in range(len(y_pred)):
    vault.append_record("sentence_transformers_mrpc_prediction", {"prediction": y_pred[i]}, 
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                           "sentence-transformers-sentence-1": [i, i + 1],
                           "sentence-transformers-sentence-2": [i, i + 1],
                       }
                       )

description = "sentence_transformers_mrpc_prediction stores the model\u2019s per-example binary predictions for the GLUE MRPC validation set. Each record corresponds to one sentence pair from glue_mrpc_validation and contains a single field, prediction, where 1 indicates the pair is predicted to be a paraphrase and 0 indicates not_paraphrase. Predictions are produced by encoding sentence1 and sentence2 separately with the sentence-transformers/all-MiniLM-L6-v2 bi-encoder, computing cosine similarity between the normalized embeddings, and applying a fixed threshold of 0.80. In this workflow, this dataset serves as the primary output of the inference step and is used downstream for evaluation, error analysis, and summary metric generation."
embedding = get_embeddings(description)
vault.create_description("sentence_transformers_mrpc_prediction", description, embedding)

properties = {"task": "paraphrase detection", "dataset_role": "model predictions", "prediction_type": "binary classification", "label_space": "0=not_paraphrase,1=paraphrase", "input_type": "sentence pair", "similarity_metric": "cosine similarity", "decision_rule": "score>=0.80", "model": "sentence-transformers/all-MiniLM-L6-v2", "split": "validation", "size": "408", "source": "glue/mrpc", "domain": "news", "language": "en"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence_transformers_mrpc_prediction", cat, embedding, prop)

{'accuracy': 0.6764705882352942, 'f1': 0.7441860465116279, 'threshold': 0.8}
                precision    recall  f1-score   support

not_paraphrase       0.49      0.65      0.56       129
    paraphrase       0.81      0.69      0.74       279

      accuracy                           0.68       408
     macro avg       0.65      0.67      0.65       408
  weighted avg       0.71      0.68      0.69       408



In [ ]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])
print({"accuracy": acc, "f1": f1, "threshold": threshold})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))

In [6]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
score: 0.9382226467132568
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
score: 0.4680331349372864
true: 0 pred: 0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
score: 0.904844343662262
true: 0 pred: 1
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will deci

In [7]:
vault.create_record_list("sentence_transformers_bi_encoder_cosine_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("sentence_transformers_bi_encoder_cosine_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "sentence_transformers_mrpc_prediction": [0, len(ds)]
                    })

summary

description = "Aggregate evaluation summary for the sentence-transformers bi-encoder cosine-similarity experiment on the GLUE MRPC validation set. This dataset contains a single record with overall validation metrics computed by comparing thresholded cosine-similarity predictions against the ground-truth paraphrase labels. Fields are: accuracy (float), f1 (float), and classification_report (string containing the full per-class sklearn classification report for not_paraphrase and paraphrase). In this workflow, it serves as the final experiment-level summary artifact, linked to the full validation dataset and the generated prediction records, so users can quickly inspect overall model performance without re-running the notebook."
embedding = get_embeddings(description)
vault.create_description("sentence_transformers_bi_encoder_cosine_mrpc_summary", description, embedding)

properties = {"task": "paraphrase detection", "dataset_role": "evaluation summary", "source": "glue/mrpc", "split": "validation", "size": "408", "domain": "news", "model_type": "sentence-transformer bi-encoder", "model_name": "sentence-transformers/all-MiniLM-L6-v2", "similarity_metric": "cosine similarity", "prediction_method": "thresholded similarity", "threshold": "0.80", "metrics": "accuracy,f1,classification_report"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence_transformers_bi_encoder_cosine_mrpc_summary", cat, embedding, prop)


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'sentence-transformers/all-MiniLM-L6-v2',
 'device': 'mps',
 'threshold': 0.8,
 'num_examples': 408,
 'accuracy': 0.6764705882352942,
 'f1': 0.7441860465116279}

In [ ]:
description = "This notebook evaluates a sentence-transformer bi-encoder approach for paraphrase detection on the GLUE MRPC validation set. It uses the sentence-transformers/all-MiniLM-L6-v2 model to encode each sentence in a pair into normalized 384-dimensional embeddings, computes cosine similarity between the two embeddings, and applies a fixed similarity threshold to predict whether the pair is a paraphrase.\n\nThe workflow loads MRPC validation examples from TableVault, generates embeddings for sentence1 and sentence2 in batches, stores those embeddings with lineage links back to the source dataset, creates binary paraphrase predictions from cosine scores, and logs the predictions to TableVault. It then computes evaluation metrics including accuracy, F1, and a classification report, inspects example predictions and errors, and stores a summary of results. The notebook also attaches natural-language descriptions and metadata embeddings to the generated artifacts and the overall process to support discoverability and documentation in TableVault." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("sentence_transformers_bi_encoder_cosine_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "problem_type": "binary text pair classification", "approach": "bi-encoder sentence embedding with cosine similarity thresholding", "model": "sentence-transformers/all-MiniLM-L6-v2", "embedding_model": "text-embedding-3-large", "dataset": "glue/mrpc", "dataset_split": "validation", "framework": "sentence-transformers, PyTorch", "similarity_metric": "cosine similarity", "threshold": "0.80", "evaluation": "accuracy, f1-score, classification report", "prediction_artifact": "sentence_transformers_mrpc_prediction", "summary_artifact": "sentence_transformers_bi_encoder_cosine_mrpc_summary", "tracking": "tablevault", "hardware": "mps_or_cpu", "input_type": "sentence pairs"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence_transformers_bi_encoder_cosine_mrpc", cat, embedding, prop)